# VinDr-PCXR - Weakly-Supervised Localization with Grad-CAM
DenseNet-121 multi-label classifier + Grad-CAM XAI evaluated against radiologist bounding boxes.

## 0. Install & Imports

In [2]:
!pip install -q grad-cam albumentations pydicom opencv-python-headless scikit-learn

!pip install -q SimpleITK

import SimpleITK as sitk

import os, json, glob, cv2, pydicom, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 19.6 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 53.8 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 

## 1. Paths & Constants

In [3]:
# ── Adjust to your Kaggle dataset paths ──────────────────────────────────────
ROOT          = Path('/kaggle/input/datasets/deenalad/sem4-data/vindr-pcxr-an-open-large-scale-pediatric-chest-x-ray-dataset-for-interpretation-of-common-thoracic-diseases-1.0.0')          # dataset root
DICOM_TRAIN   = ROOT / 'train'                             # DICOM files
DICOM_TEST    = ROOT / 'test'
PNG_TRAIN     = Path('/kaggle/working/png/train')
PNG_TEST      = Path('/kaggle/working/png/test')
ANNOT_FILE    = ROOT / 'image_labels_train.csv'
BOX_FILE      = ROOT / 'annotations_train.csv'
OUT_DIR       = Path('/kaggle/working/outputs')

IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 20
LR            = 1e-4
GRAD_CAM_THRESH = 0.4   # heatmap threshold → binary mask

for p in [PNG_TRAIN, PNG_TEST, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

## 3. Label Preparation

In [4]:
df_labels = pd.read_csv(ANNOT_FILE)

# CSV is already wide — drop rad_ID, take max vote per image across radiologists
CLASS_NAMES = [c for c in df_labels.columns if c not in ['image_id', 'rad_ID']]
NUM_CLASSES = len(CLASS_NAMES)
print(f'{NUM_CLASSES} classes:', CLASS_NAMES)

# Multiple rows per image (one per radiologist) → take max (positive if any rad flagged it)
df_wide = df_labels.groupby('image_id')[CLASS_NAMES].max().reset_index()

# Save class names
with open(OUT_DIR / 'class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

# Map image_id → DICOM path
def find_dicom(image_id, base_dir):
    matches = list(base_dir.rglob(f'{image_id}.dicom')) + list(base_dir.rglob(f'{image_id}.dcm'))
    return str(matches[0]) if matches else None

df_wide['dicom_path'] = df_wide['image_id'].apply(lambda x: find_dicom(x, DICOM_TRAIN))
df_wide = df_wide.dropna(subset=['dicom_path']).reset_index(drop=True)
print(f'Matched {len(df_wide)} images')

from sklearn.model_selection import train_test_split
df_train, df_val = train_test_split(df_wide, test_size=0.1, random_state=42)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
print(f'Train: {len(df_train)} | Val: {len(df_val)}')

15 classes: ['No finding', 'Bronchitis', 'Brocho-pneumonia', 'Other disease', 'Bronchiolitis', 'Situs inversus', 'Pneumonia', 'Pleuro-pneumonia', 'Diagphramatic hernia', 'Tuberculosis', 'Congenital emphysema', 'CPAM', 'Hyaline membrane disease', 'Mediastinal tumor', 'Lung tumor']
Matched 7728 images
Train: 6955 | Val: 773


## 4. Dataset & DataLoaders

In [5]:
class PCXRDataset(Dataset):
    def __init__(self, df, class_names, transforms=None):
        self.df          = df
        self.class_names = class_names
        self.transforms  = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Read DICOM on-the-fly
        arr = sitk.GetArrayFromImage(sitk.ReadImage(row['dicom_path'])).astype(np.float32)
        if arr.ndim == 3:
            arr = arr[0]
        
        arr = ((arr - arr.min()) / (arr.max() - arr.min() + 1e-6) * 255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        arr   = clahe.apply(arr)
        arr   = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)  # (H, W, 3)

        label = torch.tensor(row[self.class_names].values.astype(np.float32))
        if self.transforms:
            arr = self.transforms(image=arr)['image']
        return arr, label


MEAN = [0.485, 0.456, 0.406]   # ImageNet stats
STD  = [0.229, 0.224, 0.225]

train_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomResizedCrop((IMG_SIZE, IMG_SIZE), scale=(0.85, 1.0)),
    A.CLAHE(clip_limit=2.0, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

train_ds = PCXRDataset(df_train, CLASS_NAMES, train_tfm)
val_ds   = PCXRDataset(df_val,   CLASS_NAMES, val_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print('Loaders ready.')

Loaders ready.


## 5. Model — DenseNet-121 with GAP Head

In [6]:
class DenseNetPCXR(nn.Module):
    """
    DenseNet-121 backbone.
    Final conv block kept intact so GradCAM can hook into `features.denseblock4`.
    Classifier head: GlobalAveragePool → Dropout → FC(num_classes) → Sigmoid.
    """
    def __init__(self, num_classes: int, pretrained: bool = True, drop_rate: float = 0.5):
        super().__init__()
        base           = models.densenet121(weights='IMAGENET1K_V1' if pretrained else None)
        self.features  = base.features            # all conv blocks
        self.relu      = nn.ReLU(inplace=True)
        self.gap       = nn.AdaptiveAvgPool2d(1)  # Global Average Pool
        self.dropout   = nn.Dropout(p=drop_rate)
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, x):
        feat = self.features(x)          # (B, 1024, 7, 7)
        feat = self.relu(feat)
        out  = self.gap(feat)            # (B, 1024, 1, 1)
        out  = torch.flatten(out, 1)     # (B, 1024)
        out  = self.dropout(out)
        return self.classifier(out)      # (B, num_classes) — raw logits


model = DenseNetPCXR(num_classes=NUM_CLASSES).to(DEVICE)

# GradCAM target layer = last conv layer in denseblock4
TARGET_LAYER = model.features.denseblock4
print('Model ready. Target layer for GradCAM:', TARGET_LAYER.__class__.__name__)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 118MB/s] 


Model ready. Target layer for GradCAM: _DenseBlock


## 6. Class-Weighted Loss & Optimiser

In [7]:
# Positive-weight per class: neg_count / pos_count (down-weights majority negatives)
label_counts = df_train[CLASS_NAMES].sum(axis=0)
neg_counts   = len(df_train) - label_counts
pos_weight   = torch.tensor((neg_counts / label_counts.clip(lower=1)).values, dtype=torch.float32).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print('Pos weights (top-5):', pos_weight[:5].cpu().numpy().round(2))

Pos weights (top-5): [ 0.5   8.22 12.69 17.9  14.39]


## 7. Training Loop

In [8]:
def compute_mean_auc(labels_all, preds_all):
    """Per-class AUC, skip classes with only one label present."""
    aucs = []
    for i in range(labels_all.shape[1]):
        if len(np.unique(labels_all[:, i])) > 1:
            aucs.append(roc_auc_score(labels_all[:, i], preds_all[:, i]))
    return float(np.mean(aucs)), aucs


best_auc   = 0.0
history    = {'train_loss': [], 'val_loss': [], 'val_auc': []}

for epoch in range(1, EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [train]', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_ds)

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval()
    val_loss    = 0.0
    all_labels  = []
    all_preds   = []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} [val]', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            val_loss += criterion(logits, labels).item() * imgs.size(0)
            all_labels.append(labels.cpu().numpy())
            all_preds.append(torch.sigmoid(logits).cpu().numpy())
    val_loss /= len(val_ds)

    all_labels = np.concatenate(all_labels)
    all_preds  = np.concatenate(all_preds)
    mean_auc, _ = compute_mean_auc(all_labels, all_preds)

    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(mean_auc)

    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | mean_AUC={mean_auc:.4f}')

    if mean_auc > best_auc:
        best_auc = mean_auc
        torch.save(model.state_dict(), OUT_DIR / 'best_model.pth')
        print(f'  ✓ Saved best model (AUC={best_auc:.4f})')

print(f'\nTraining complete. Best val mean-AUC: {best_auc:.4f}')

Epoch 1/20 [train]:   0%|          | 0/218 [00:00<?, ?it/s]

Epoch 1/20 [val]:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 01 | train_loss=1.7426 | val_loss=0.7613 | mean_AUC=0.6836
  ✓ Saved best model (AUC=0.6836)


Epoch 2/20 [train]:   0%|          | 0/218 [00:00<?, ?it/s]

Epoch 2/20 [val]:   0%|          | 0/25 [00:00<?, ?it/s]

...

Epoch 19/20 [val]:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 19 | train_loss=0.4779 | val_loss=0.5460 | mean_AUC=0.8117
  ✓ Saved best model (AUC=0.8117)


Epoch 20/20 [train]:   0%|          | 0/218 [00:00<?, ?it/s]

Epoch 20/20 [val]:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 20 | train_loss=0.4818 | val_loss=0.5637 | mean_AUC=0.8117
  ✓ Saved best model (AUC=0.8117)

Training complete. Best val mean-AUC: 0.8117


## 8. Per-Class AUC Report

In [9]:
model.load_state_dict(torch.load(OUT_DIR / 'best_model.pth'))
model.eval()

all_labels, all_preds = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        logits = model(imgs.to(DEVICE))
        all_labels.append(labels.numpy())
        all_preds.append(torch.sigmoid(logits).cpu().numpy())

all_labels = np.concatenate(all_labels)
all_preds  = np.concatenate(all_preds)

results = []
for i, cls in enumerate(CLASS_NAMES):
    if len(np.unique(all_labels[:, i])) > 1:
        auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
        results.append({'class': cls, 'AUC': round(auc, 4), 'pos_samples': int(all_labels[:, i].sum())})

df_results = pd.DataFrame(results).sort_values('AUC', ascending=False)
print(df_results.to_string(index=False))
print(f"\nMean AUC: {df_results['AUC'].mean():.4f}")

                   class    AUC  pos_samples
            Tuberculosis 0.9922            2
          Situs inversus 0.9922            1
Hyaline membrane disease 0.9514            2
               Pneumonia 0.7896           51
        Brocho-pneumonia 0.7468           37
              No finding 0.7295          520
           Other disease 0.7290           44
              Bronchitis 0.6887           88
           Bronchiolitis 0.6864           45

Mean AUC: 0.8118


## 9. Grad-CAM Heatmap Generation

In [19]:
class SemanticTarget:
    def __init__(self, class_idx): self.class_idx = class_idx
    def __call__(self, output): return output[self.class_idx]  # remove [0, ...]


def get_gradcam_heatmap(model, img_tensor, class_idx, target_layer):
    """
    Returns: heatmap (H, W) float [0,1]
    """
    cam = GradCAM(model=model, target_layers=[target_layer])
    grayscale_cam = cam(
        input_tensor=img_tensor.unsqueeze(0).to(DEVICE),
        targets=[SemanticTarget(class_idx)],
        aug_smooth=True,
        eigen_smooth=True
    )
    return grayscale_cam[0]   # (H, W)


def overlay_heatmap(original_img_np, heatmap):
    """Blend heatmap onto image. original_img_np: (H,W,3) uint8."""
    img_float = original_img_np.astype(np.float32) / 255.0
    return show_cam_on_image(img_float, heatmap, use_rgb=True)


print('GradCAM utilities ready.')

GradCAM utilities ready.


## 10. XAI Evaluation: IoU & Pointing Game vs. Radiologist Boxes

In [20]:
# Image-level eval: does the top-predicted class heatmap overlap ANY GT box?
model.eval()
eval_records = []

val_ids_with_boxes = set(df_boxes['image_id'].unique()) & set(df_val['image_id'].unique())
print(f'Val images with GT boxes: {len(val_ids_with_boxes)}')

for image_id in tqdm(list(val_ids_with_boxes)[:SAMPLE_LIMIT], desc='XAI eval'):
    row = df_val[df_val['image_id'] == image_id].iloc[0]
    
    arr = sitk.GetArrayFromImage(sitk.ReadImage(row['dicom_path'])).astype(np.float32)
    if arr.ndim == 3: arr = arr[0]
    arr = ((arr - arr.min()) / (arr.max() - arr.min() + 1e-6) * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_224 = cv2.cvtColor(cv2.resize(clahe.apply(arr), (IMG_SIZE, IMG_SIZE)), cv2.COLOR_GRAY2RGB)
    orig_h, orig_w = arr.shape[:2]

    tensor = val_tfm(image=img_224)['image']
    with torch.no_grad():
        probs = torch.sigmoid(model(tensor.unsqueeze(0).to(DEVICE))).cpu().numpy()[0]

    # Evaluate top-3 predicted classes
    top_indices = np.argsort(probs)[::-1][:3]
    boxes_this  = df_boxes[df_boxes['image_id'] == image_id]
    sx, sy      = IMG_SIZE / orig_w, IMG_SIZE / orig_h

    # Union of all GT boxes for this image (any finding counts)
    gt_boxes_224 = [
        [int(r.x_min*sx), int(r.y_min*sy), int(r.x_max*sx), int(r.y_max*sy)]
        for _, r in boxes_this.iterrows()
    ]

    for cls_idx in top_indices:
        heatmap  = get_gradcam_heatmap(model, tensor, cls_idx, TARGET_LAYER)
        pred_box = heatmap_to_bbox(heatmap)

        best_iou = max((compute_iou(pred_box, gb) for gb in gt_boxes_224), default=0.0) if pred_box else 0.0
        best_pg  = max((pointing_game(heatmap, gb) for gb in gt_boxes_224), default=0)

        eval_records.append({
            'image_id': image_id,
            'class': CLASS_NAMES[cls_idx],
            'prob': float(probs[cls_idx]),
            'iou': best_iou,
            'pointing_game': best_pg
        })

df_eval = pd.DataFrame(eval_records)
summary = (
    df_eval.groupby('class')
    .agg(mean_IoU=('iou','mean'), pointing_game_acc=('pointing_game','mean'), n=('iou','count'))
    .sort_values('mean_IoU', ascending=False)
    .round(4)
)
print('\n=== XAI Evaluation Table ===')
print(summary.to_string())
print(f"\nOverall mean IoU         : {df_eval['iou'].mean():.4f}")
print(f"Overall Pointing Game Acc: {df_eval['pointing_game'].mean():.4f}")
summary.to_csv(OUT_DIR / 'xai_eval_table.csv')

Val images with GT boxes: 253


XAI eval:   0%|          | 0/253 [00:00<?, ?it/s]


=== XAI Evaluation Table ===
                          mean_IoU  pointing_game_acc    n
class                                                     
Tuberculosis                0.2133             0.3333   12
Hyaline membrane disease    0.2118             0.4286   14
Congenital emphysema        0.1527             1.0000    1
Pneumonia                   0.1406             0.2209   86
Other disease               0.1134             0.2195  123
No finding                  0.1039             0.2632  133
Bronchiolitis               0.1037             0.2632  114
Brocho-pneumonia            0.0923             0.1698  106
Lung tumor                  0.0879             0.5000    2
Bronchitis                  0.0806             0.1757  148
Mediastinal tumor           0.0730             0.3333    3
CPAM                        0.0483             0.1250    8
Situs inversus              0.0123             0.0000    9

Overall mean IoU         : 0.1054
Overall Pointing Game Acc: 0.2227


## 11. Save Sample Heatmap PNGs

In [23]:
HEATMAP_DIR = OUT_DIR / 'sample_heatmaps'
HEATMAP_DIR.mkdir(exist_ok=True)

N_SAMPLES = 10
saved     = 0

for image_id in list(val_ids_with_boxes)[:50]:
    if saved >= N_SAMPLES: break
    row = df_val[df_val['image_id'] == image_id].iloc[0]

    # Read DICOM on-the-fly
    arr = sitk.GetArrayFromImage(sitk.ReadImage(row['dicom_path'])).astype(np.float32)
    if arr.ndim == 3: arr = arr[0]
    orig_h, orig_w = arr.shape[:2]
    arr = ((arr - arr.min()) / (arr.max() - arr.min() + 1e-6) * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_224 = cv2.cvtColor(cv2.resize(clahe.apply(arr), (IMG_SIZE, IMG_SIZE)), cv2.COLOR_GRAY2RGB)

    tensor = val_tfm(image=img_224)['image']
    with torch.no_grad():
        probs = torch.sigmoid(model(tensor.unsqueeze(0).to(DEVICE))).cpu().numpy()[0]

    top_idx  = int(np.argmax(probs))
    top_cls  = CLASS_NAMES[top_idx]
    top_prob = probs[top_idx]
    heatmap  = get_gradcam_heatmap(model, tensor, top_idx, TARGET_LAYER)
    overlay  = overlay_heatmap(img_224, heatmap)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(f'{image_id}  |  {top_cls} ({top_prob:.1%})', fontsize=11)
    axes[0].imshow(img_224);             axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(heatmap, cmap='jet'); axes[1].set_title('Grad-CAM'); axes[1].axis('off')
    axes[2].imshow(overlay);             axes[2].set_title('Overlay');  axes[2].axis('off')

    sx, sy = IMG_SIZE / orig_w, IMG_SIZE / orig_h
    for _, box_row in df_boxes[df_boxes['image_id'] == image_id].iterrows():
        rect = patches.Rectangle(
            (box_row['x_min']*sx, box_row['y_min']*sy),
            (box_row['x_max']-box_row['x_min'])*sx,
            (box_row['y_max']-box_row['y_min'])*sy,
            linewidth=2, edgecolor='lime', facecolor='none'
        )
        axes[2].add_patch(rect)
        axes[2].text(box_row['x_min']*sx, box_row['y_min']*sy - 3,
                     box_row['class_name'], color='lime', fontsize=7)

    plt.tight_layout()
    plt.savefig(HEATMAP_DIR / f'{image_id}_{top_cls}.png', dpi=100, bbox_inches='tight')
    plt.close()
    saved += 1

print(f'Saved {saved} sample heatmap PNGs → {HEATMAP_DIR}')

print('\n=== Artifacts ===')
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file(): print(' ', f.relative_to(OUT_DIR))

Saved 10 sample heatmap PNGs → /kaggle/working/outputs/sample_heatmaps

=== Artifacts ===
  best_model.pth
  class_names.json
  sample_heatmaps/1cc9bb3024da52246c21fc3fea499adf_Bronchiolitis.png
  sample_heatmaps/22bc83890b50718002839691bf5d3fff_No finding.png
  sample_heatmaps/2c7021918e85939562b59b4a9cf60969_No finding.png
  sample_heatmaps/6d1f4dafdb6f518fbcdda722e9468c9c_Mediastinal tumor.png
  sample_heatmaps/760ca6b5e84c0a6189422a5ef7a2ca0e_No finding.png
  sample_heatmaps/7ad8cbfa5e5cb5d9541a9f9a7219a9d9_No finding.png
  sample_heatmaps/81cfd78cb555521b73c34e07f0fff829_No finding.png
  sample_heatmaps/89617377f7c5d1c8ad810519b58ccb7e_Situs inversus.png
  sample_heatmaps/9822adb6089d30b0e0750a986d40d01a_No finding.png
  sample_heatmaps/c497698abda26cfd34bac840585e5be2_Bronchitis.png
  xai_eval_table.csv
